# Diagnostic Feature Selection

⚠️  **IMPORTANT: This notebook is for DIAGNOSTIC VALIDATION only**

**Purpose:** Validate theory-driven variable selection for PSM-DiD, NOT to generate primary model specifications.

**Key Principle:** Econometric models are specified from KNOWLEDGE (causal theory), not from DATA (statistical filters).

**Workflow:**
1. Define your theory-driven specification
2. Run diagnostic validation:
   - Check multicollinearity (VIF)
   - Compare with auto-selected features
   - Verify data quality
3. Use theory-driven specification in PSM-DiD (NOT selected_features)
4. (Optional) Robustness check with auto-selected features

**Diagnostic Methods:**
1. Multicollinearity diagnostic (VIF for your variables)
2. Specification validation (overlap, importance, quality)
3. Specification comparison (theory-driven vs auto-selected)

In [ ]:
# Import and setup
from asean_green_bonds import data, config
from asean_green_bonds.data.feature_selection import (
    diagnose_multicollinearity,
    validate_specification,
    compare_specifications
)
import pandas as pd
import numpy as np

print('Loading engineered data...')
df = data.load_processed_data(which='engineered')
print(f'Data shape: {df.shape}')
print(f'Columns: {df.shape[1]}')

In [ ]:
# STEP 1: DEFINE YOUR THEORY-DRIVEN SPECIFICATION
# These variables should be chosen based on causal theory, not data

# Core control variables for PSM specification
psm_vars = [
    'Firm_Size',                    # Firm size affects issuance capacity
    'Leverage',                     # Debt levels affect market access
    'Return_on_Assets',             # Profitability affects financing costs
    'Asset_Tangibility',            # Tangible assets as collateral
    'Prior_Green_Bonds',            # Track record signals commitment
    'Issuer_Track_Record',          # Number of prior green bond issues
    'Has_Green_Framework',          # Documented green bond framework
]

# DiD controls: Pre-treatment characteristics for parallel trends
did_controls = psm_vars + [
    'Lagged_ESG_Score',             # Pre-treatment environmental profile
    'Lagged_Return_on_Assets',      # Pre-treatment performance trend
    'Lagged_Leverage',              # Pre-treatment financial structure
]

# Verify variables exist in data
available_psm = [v for v in psm_vars if v in df.columns]
available_did = [v for v in did_controls if v in df.columns]

print(f'PSM variables available: {len(available_psm)}/{len(psm_vars)}')
print(f'Missing: {set(psm_vars) - set(available_psm)}')
print(f'\nDiD controls available: {len(available_did)}/{len(did_controls)}')

In [ ]:
# STEP 2: VALIDATE YOUR PSM SPECIFICATION WITH DIAGNOSTICS

print('='*70)
print('DIAGNOSTIC VALIDATION: PSM SPECIFICATION')
print('='*70)

# Run comprehensive diagnostic
psm_report = validate_specification(
    df,
    theory_vars=available_psm,
    outcome_col='ESG_Score',
    control_cols=available_psm
)

print(psm_report)

In [ ]:
# STEP 3: CHECK MULTICOLLINEARITY FOR YOUR SPECIFICATION

print('\n' + '='*70)
print('MULTICOLLINEARITY CHECK: VIF FOR PSM VARIABLES')
print('='*70)

vif_psm = diagnose_multicollinearity(df, available_psm)
print(vif_psm.to_string())

print('\nInterpretation:')
print('  ✓ OK: VIF < 5')
print('  ⚠️  Warning: VIF 5-10')
print('  ❌ High: VIF > 10')

high_vif = vif_psm[vif_psm['VIF'] > 10]
if len(high_vif) > 0:
    print(f'\n⚠️  WARNING: {len(high_vif)} variables have VIF > 10:')
    print(high_vif[['Variable', 'VIF', 'Status']])
else:
    print('\n✅ No multicollinearity issues detected')

In [ ]:
# STEP 4: COMPARE THEORY-DRIVEN VS AUTO-SELECTED FEATURES
# This shows how your specification aligns with data patterns

print('\n' + '='*70)
print('SPECIFICATION COMPARISON')
print('='*70)

comparison = compare_specifications(df, available_psm, 'ESG_Score')
print(comparison.to_string())

print('\nInterpretation:')
print('  ✓ Your variables in auto-selected = Good data-theory alignment')
print('  ⚠️  Your variables NOT in auto-selected = Low-signal but essential confounders')
print('  📊 Extra auto-selected vars = Potential robustness check candidates')

In [ ]:
# STEP 5: (OPTIONAL) PREPARE FOR ROBUSTNESS CHECK
# Load auto-selected features for comparison in PSM-DiD results

print('\n' + '='*70)
print('ROBUSTNESS CHECK PREPARATION')
print('='*70)

print('\nTo conduct robustness check with auto-selected features:')
print('\n1. In 03_methodology_and_results.ipynb, add this cell:')
print('   ```')
print('   # Main specification (theory-driven)')
print('   main_results = analysis.run_psm_did(df_eng, ps_vars=available_psm, ...)')
print('   ')
print('   # Robustness check (auto-selected)')
print('   df_selected = load_processed_data(which="selected_features")')
print('   robust_results = analysis.run_psm_did(df_selected, ps_vars=selected_features, ...)')
print('   ')
print('   # Compare effects')
print('   print(f"Effect difference: {abs(main_results[\"ate\"] - robust_results[\"ate\"])}")')
print('   ```')

print('\n2. This demonstrates your results are robust to variable selection method')
print('\n✅ This approach ensures:')
print('   - Primary results use theory-driven specification')
print('   - Robustness shown through alternative specs')
print('   - Full methodological transparency')


## Summary

### What This Diagnostic Validated
✅ Your theory-driven PSM specification is appropriate
✅ No critical multicollinearity issues (all VIF < 10)
✅ Theory variables align with data patterns
✅ Data quality sufficient for analysis

### What To Do Next
1. **Use engineered data** (NOT selected_features) in 03_methodology
2. **Use available_psm** (your theory-driven variables) in PSM-DiD
3. **Use available_did** (+ lagged controls) in DiD specification
4. **(Optional)** Add robustness check with auto-selected features

### Important Reminder
- ✅ DO: Use engineered data with manual variable selection
- ✅ DO: Run diagnostics to validate your specification
- ✅ DO: Include robustness checks in appendix
- ❌ DON'T: Use selected_features for primary PSM-DiD model
- ❌ DON'T: Let data-driven filters override causal theory

### References
See `DIAGNOSTIC_FEATURE_SELECTION_GUIDE.md` for complete documentation on diagnostic use of feature selection.